In [1]:
!pip install  -q torch torchvision opencv-python tqdm matplotlib

In [3]:
import os, json, math, random
from typing import Optional, Tuple

import numpy as np
import cv2
from tqdm import tqdm
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
import torchvision

In [4]:
# Download FreiHAND
!wget https://lmb.informatik.uni-freiburg.de/data/freihand/FreiHAND_pub_v2.zip
!unzip -q FreiHAND_pub_v2.zip -d /content/FreiHAND

--2025-12-14 03:32:51--  https://lmb.informatik.uni-freiburg.de/data/freihand/FreiHAND_pub_v2.zip
Resolving lmb.informatik.uni-freiburg.de (lmb.informatik.uni-freiburg.de)... 132.230.167.23
Connecting to lmb.informatik.uni-freiburg.de (lmb.informatik.uni-freiburg.de)|132.230.167.23|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 3881833583 (3.6G) [application/zip]
Saving to: ‘FreiHAND_pub_v2.zip’

FreiHAND_pub_v2.zip 100%[===================>]   3.62G  28.1MB/s    in 2m 19s  

2025-12-14 03:35:11 (26.6 MB/s) - ‘FreiHAND_pub_v2.zip’ saved [3881833583/3881833583]



In [ ]:
# CONFIG
FREIHAND_ROOT = "/content/FreiHAND"

RGB_DIR = os.path.join(FREIHAND_ROOT, "training", "rgb")
XYZ_JSON = os.path.join(FREIHAND_ROOT, "training", "training_xyz.json")
K_JSON = os.path.join(FREIHAND_ROOT, "training", "training_K.json")

# image/heatmap size
IN_SIZE = 224
HM_SIZE = 56
SIGMA = 2.0

# training
BATCH_SIZE = 32
EPOCHS = 50
LR = 3e-4
WEIGHT_DECAY = 1e-4
NUM_WORKERS = 4

# split
VAL_RATIO = 0.1
SEED = 42

# crop margin
CROP_MARGIN = 0.25

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")

# Sanity checks
assert os.path.isdir(RGB_DIR), f"RGB_DIR not found: {RGB_DIR}"
assert os.path.isfile(XYZ_JSON), f"XYZ_JSON not found: {XYZ_JSON}"
assert os.path.isfile(K_JSON),   f"K_JSON not found: {K_JSON}"